# WikiTrend Gold Inspection

This notebook inspects Gold Parquet tables with DuckDB. It reads bounded samples and aggregate checks without loading the full dataset into pandas.

## 1. Project setup

Run this notebook from the repository root or from the `notebooks` directory. The notebook prefers canonical `data/gold`; while a build is staged, it can inspect `data/gold_hardened`.

In [ ]:
from pathlib import Path

import duckdb
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists() and (PROJECT_ROOT.parent / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

CANONICAL_GOLD_DIR = PROJECT_ROOT / 'data' / 'gold'
STAGING_GOLD_DIR = PROJECT_ROOT / 'data' / 'gold_hardened'
if CANONICAL_GOLD_DIR.exists() and any(CANONICAL_GOLD_DIR.rglob('*.parquet')):
    GOLD_DIR = CANONICAL_GOLD_DIR
elif STAGING_GOLD_DIR.exists():
    GOLD_DIR = STAGING_GOLD_DIR
else:
    GOLD_DIR = CANONICAL_GOLD_DIR

TABLES = [
    'page_hourly',
    'hourly_project_traffic',
    'top_pages_hourly',
    'modeling_page_hourly',
    'trending_pages',
    'anomaly_alerts',
    'forecast_features',
    'forecast_evaluation',
]

con = duckdb.connect()

def table_files(table):
    return sorted((GOLD_DIR / table).rglob('*.parquet'))

def table_exists(table):
    return bool(table_files(table))

def table_source(table):
    return f"read_parquet('{(GOLD_DIR / table / '**' / '*.parquet').as_posix()}')"

print(f'Project root: {PROJECT_ROOT}')
print(f'Gold directory: {GOLD_DIR}')
print(f'Gold directory exists: {GOLD_DIR.exists()}')

## 2. Gold file manifest

A manifest tells us which tables are present, how many Parquet files they contain, and whether a table is still being written.

In [ ]:
manifest_rows = []
for table in TABLES:
    files = table_files(table)
    manifest_rows.append({
        'table': table,
        'exists': (GOLD_DIR / table).exists(),
        'parquet_files': len(files),
        'complete_marker': (GOLD_DIR / table / '_SUCCESS').exists(),
        'bytes': sum(path.stat().st_size for path in files),
    })

manifest = pd.DataFrame(manifest_rows)
manifest['gigabytes'] = manifest['bytes'] / 1024**3
display(manifest)

## 3. Table schemas

Inspect each schema before writing queries. This catches contract changes such as renamed rolling-window or forecast columns.

In [ ]:
for table in TABLES:
    if not table_exists(table):
        print(f'{table}: no Parquet files yet')
        continue
    print(f'--- {table} ---')
    display(con.execute(f'DESCRIBE SELECT * FROM {table_source(table)}').df())

## 4. Row counts and coverage

These checks show the time window and project coverage represented in each time-grained Gold table.

In [ ]:
coverage_rows = []
for table in ['page_hourly', 'hourly_project_traffic', 'top_pages_hourly', 'trending_pages', 'anomaly_alerts', 'forecast_features']:
    if not table_exists(table):
        continue
    row = con.execute(f'''
        SELECT count(*) AS rows, min(date) AS first_date, max(date) AS last_date,
               count(DISTINCT hour) AS hours, count(DISTINCT project) AS projects, count(DISTINCT access_mode) AS access_modes
        FROM {table_source(table)}
    ''').fetchone()
    coverage_rows.append({'table': table, 'rows': row[0], 'first_date': row[1], 'last_date': row[2], 'hours': row[3], 'projects': row[4], 'access_modes': row[5]})

display(pd.DataFrame(coverage_rows))

## 5. Inspect `page_hourly`

This is the topic-level Gold aggregate. The analytical key is `(date, hour, project, access_mode, normalized_title)`. `page_title` is retained as a representative raw title for auditability.

In [ ]:
if table_exists('page_hourly'):
    display(con.execute(f'''
        SELECT date, hour, source_project, project, language, project_family, access_mode, page_title, normalized_title,
               view_count, response_size, page_rows
        FROM {table_source('page_hourly')}
        WHERE date = DATE '2026-08-01' AND hour = 0 AND project = 'en' AND access_mode = 'desktop'
        ORDER BY view_count DESC, normalized_title
        LIMIT 25
    ''').df())

## 6. Inspect top pages

Ranks are calculated independently for each project, access mode, and hour.

In [ ]:
if table_exists('top_pages_hourly'):
    display(con.execute(f'''
        SELECT timestamp_hour, project, access_mode, rank, normalized_title, view_count, page_rows
        FROM {table_source('top_pages_hourly')}
        ORDER BY timestamp_hour DESC, project, rank
        LIMIT 50
    ''').df())

## 7. Inspect trending topics

The trend score uses a past-only `robust_z_score` based on log-space median and MAD, log-scaled current volume, and baseline coverage. The current hour is excluded from the rolling baseline.

In [ ]:
if table_exists('trending_pages'):
    display(con.execute(f'''
        SELECT timestamp_hour, project, access_mode, normalized_title, view_count, previous_hour_views,
               rolling_baseline_avg, rolling_baseline_stddev,
               rolling_baseline_log_median, rolling_baseline_log_mad,
               baseline_observed_hours, growth_rate, robust_z_score,
               trend_score, trend_rank
        FROM {table_source('trending_pages')}
        WHERE robust_z_score IS NOT NULL
        ORDER BY timestamp_hour DESC, trend_score DESC
        LIMIT 50
    ''').df())

## 8. Inspect anomaly alerts

Alerts are high-volume rows whose `robust_z_score` crosses the configured threshold.

In [ ]:
if table_exists('anomaly_alerts'):
    alert_summary = con.execute(f'''
        SELECT date, hour, project, access_mode, alert_severity, count(*) AS alerts, max(robust_z_score) AS max_robust_z_score
        FROM {table_source('anomaly_alerts')}
        GROUP BY 1, 2, 3, 4, 5
        ORDER BY date DESC, hour DESC, alerts DESC
    ''').df()
    display(alert_summary)
    display(con.execute(f'''
        SELECT timestamp_hour, project, access_mode, normalized_title, view_count, robust_z_score, trend_score, alert_severity
        FROM {table_source('anomaly_alerts')}
        ORDER BY timestamp_hour DESC, robust_z_score DESC
        LIMIT 50
    ''').df())
else:
    print('No anomaly Parquet files yet; an empty alert table is valid.')

## 9. Inspect forecast features

`target_next_hour_views` is an evaluation label. It must not be used as an input feature when producing a live forecast.

In [ ]:
if table_exists('forecast_features'):
    display(con.execute(f'''
        SELECT timestamp_hour, project, access_mode, normalized_title, is_observed, view_count, lag_1h_views, lag_24h_views,
               rolling_forecast_avg, forecast_history_elapsed_hours, baseline_forecast,
               forecast_available, target_next_hour_views
        FROM {table_source('forecast_features')}
        WHERE forecast_available
        ORDER BY timestamp_hour DESC, baseline_forecast DESC
        LIMIT 50
    ''').df())

## 10. Forecast evaluation

Compare the leakage-safe baseline methods. Lower MASE, ND, sMAPE, and msMAPE are better. 
sMAPE handles zero/zero as zero; msMAPE uses an epsilon-stabilized denominator.

In [ ]:
if table_exists('forecast_evaluation'):
    display(con.execute(f'''
        SELECT project, access_mode, forecast_method, evaluated_rows, mase_valid_rows, mase, nd, smape, msmape,
               evaluation_start_hour, evaluation_end_hour
        FROM {table_source('forecast_evaluation')}
        ORDER BY project, mase NULLS LAST
    ''').df())
else:
    print('Forecast evaluation is not committed yet.')

## 11. Gold quality checks

These checks should return zero for invalid values and duplicate keys.

In [ ]:
quality_checks = {}
if table_exists('page_hourly'):
    source = table_source('page_hourly')
    quality_checks['page_hourly_duplicate_keys'] = con.execute(f'''
        SELECT count(*) FROM (
            SELECT date, hour, project, access_mode, normalized_title
            FROM {source}
            GROUP BY 1, 2, 3, 4, 5 HAVING count(*) > 1
        )
    ''').fetchone()[0]
    quality_checks['page_hourly_invalid_views'] = con.execute(f'''
        SELECT count(*) FROM {source} WHERE view_count IS NULL OR view_count < 0
    ''').fetchone()[0]
    quality_checks['page_hourly_blank_titles'] = con.execute(f'''
        SELECT count(*) FROM {source} WHERE normalized_title IS NULL OR trim(normalized_title) = ''
    ''').fetchone()[0]
if table_exists('top_pages_hourly'):
    quality_checks['top_pages_duplicate_keys'] = con.execute(f'''
        SELECT count(*) - count(DISTINCT concat(cast(date AS VARCHAR), '|', cast(hour AS VARCHAR), '|', project, '|', access_mode, '|', normalized_title))
        FROM {table_source('top_pages_hourly')}
    ''').fetchone()[0]
if table_exists('forecast_features'):
    quality_checks['forecast_invalid_horizon'] = con.execute(f'''
        SELECT count(*) FROM {table_source('forecast_features')} WHERE forecast_horizon_hours <> 1
    ''').fetchone()[0]

display(pd.DataFrame([{'check': key, 'value': value} for key, value in quality_checks.items()]))

## 12. Reconcile project traffic

The project aggregate should equal the sum of its topic-level rows for every date, hour, project, and access mode.

In [ ]:
if table_exists('page_hourly') and table_exists('hourly_project_traffic'):
    page_source = table_source('page_hourly')
    traffic_source = table_source('hourly_project_traffic')
    reconciliation = con.execute(f'''
        WITH pages AS (
            SELECT date, hour, project, access_mode, sum(view_count) AS view_count, sum(response_size) AS response_size
            FROM {page_source}
            GROUP BY 1, 2, 3, 4
        ), traffic AS (
            SELECT date, hour, project, access_mode, view_count, response_size
            FROM {traffic_source}
        )
        SELECT count(*) AS mismatches
        FROM pages p FULL OUTER JOIN traffic t USING (date, hour, project, access_mode)
        WHERE p.view_count IS NULL OR t.view_count IS NULL
           OR p.view_count <> t.view_count OR p.response_size <> t.response_size
    ''').df()
    display(reconciliation)

## 13. Practical inspection notes

- Start with the manifest and schema cells.
- Use `WHERE date`, `hour`, and `project` filters for bounded samples.
- Treat `target_next_hour_views` as a label, not a live feature.
- Do not delete Silver or raw files until the retention decision below has been recorded.